# Lesson 09 — 短脉冲来了，Polling 会漏掉吗？

这是一段真实的 C++ 实验：你修改代码格，再运行，就会看到下方深色虚拟设备的输入、输出和检查结果变化。Python 只负责让 VS Code Notebook 启动 C++ 编译和实验台。

## 先运行

按顺序运行初始化格和后面的代码格。绿色 ✓ 表示本格 C++ 符合当前契约；红色 ✗ 会给出预期与实际输出。


In [ ]:
# 运行支架：只运行一次，不需要学习或修改 Python。
from pathlib import Path
project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "tools/notebook_setup.py").is_file()), None)
if project is None:
    raise RuntimeError("请用 VS Code 打开 esp32-hands-on-academy 文件夹")
get_ipython().run_line_magic("run", str(project / "tools/notebook_setup.py"))


一个短脉冲可能在下一次 Polling 前已经消失。中断（Interrupt）让硬件先把事件计数，主循环随后消费计数。

## 最小知识

本机模型把 `input.pulses` 视为 ISR 已经累积的单调脉冲数。Controller 不假装在 ISR 中做复杂工作，只计算增量并更新事件数。


## 1. 第一个实验（Guided）

先运行。不要急着读每一行；先操作实验台，观察输入与输出之间的关系。


In [ ]:
%%academy_lab 09-interrupts guided guided pass
#pragma once
#include "academy/device.hpp"
namespace academy {class Controller{public:explicit Controller(Device&b):b_(b){}void tick(){b_.output.events=b_.input.pulses;}private:Device&b_;};}


## 2. 修改一次

让 LED 每三个脉冲翻转一次。思考为什么计数器比 bool 标志更不容易丢事件。

重新运行这一个格，再在实验台确认你的预测。


In [ ]:
%%academy_lab 09-interrupts modify
#pragma once
#include "academy/device.hpp"
namespace academy {class Controller{public:explicit Controller(Device&b):b_(b){}void tick(){b_.output.events=b_.input.pulses;}private:Device&b_;};}


## 3. 故障实验（Debug）

下面代码发现计数变化时只加一，两个脉冲在两次 tick 之间到达会被丢掉。运行后失败是预期现象。用面板的时间线、波形、事务或终端找出失败发生在何时。


In [ ]:
%%academy_lab 09-interrupts broken exercise fail
#pragma once
#include "academy/device.hpp"
namespace academy {class Controller{public:explicit Controller(Device&b):b_(b){}void tick(){b_.output.events=b_.input.pulses;}private:Device&b_;};}


## 4. 你来完成（Build）

保存上一次脉冲总数，消费差值。上电时只建立 baseline；计数倒退时报告 error。

默认这格会失败，失败信息是需求的一部分。先自己尝试；需要时再看 [分层 Hint](../lessons/09-interrupts/hints.md)。


In [ ]:
%%academy_lab 09-interrupts exercise exercise fail
#pragma once
#include "academy/device.hpp"
namespace academy {class Controller{public:explicit Controller(Device&b):b_(b){}void tick(){// TODO: 消费 pulses 的增量
}private:Device&b_;};}


## 5. 开放挑战

设计一个无锁 ring buffer，让 ISR 记录带时间戳的边沿。哪些操作绝不能放进 ISR？

## 你刚刚真正学到了什么？

Interrupt 的作用是降低事件丢失风险，不是让任何代码都在 ISR 中运行。主循环仍负责把事件变成业务行为。

## 硬件迁移

STEP 2 用 GPIO interrupt 只递增 `volatile`/原子计数或发送通知；去抖、优先级和 ISR 限制需在真实芯片上验证。

完成自己的尝试后才查看 [独立 Solution Notebook](solutions/09-interrupts-solution.ipynb)。
